<h1>Data Ingestion and Training By: G5-Rita Group</h1><i>notebook edition</i><br>
1). Obtain<br>
2). Decompress<br>
3). Parse<br>
<li>
<ol>A) Load Game</ol>
<ol>B) Parse Move</ol>
<ol>C) Load Move</ol>
</li>
4). Cleanup

In [1]:
#Define connection to MYSQL Server
ConnectString = "mysql -h database-1.clqxqhhe6wft.us-east-1.rds.amazonaws.com -P 3306 -u admin -p'<Enter_DB_Password>' --ssl-verify-server-cert  --ssl-ca=/certs/global-bundle.pem mysql"

host=ConnectString[9:60]
user='admin'
password='Data608-Project'
database='CHESSBOT'


In [2]:
#Define file to load into Database

#Provide two-digit month   (01 -> 12)
File_month = "05"

#Provide four-digit year  (2026  <- 2013 )
File_year = "2013"


In [3]:
### Check if file was loaded already  else load stub and obtain id for latter
import mysql.connector
import files as fs
metric = []
successbit = False
CompressedFileName = f"lichess_db_standard_rated_{File_year}-{File_month}.pgn.zst"
Files_id = -1

#connect to SQL server
print("Attempting MySQL Server connection")
connection = mysql.connector.connect(host=host,user=user,password=password,database=database,autocommit=True) 

if connection.is_connected():
    cursor = connection.cursor()
    print("Connected to MySQL Server: ")
    Files_id = fs.files(CompressedFileName,File_month,File_year,cursor)
    if(Files_id == -1):
        print("File: ",CompressedFileName," already loaded")
        STOP()
else:
    print("Failed connection to MySQL Server")
    exit()

Attempting MySQL Server connection
Connected to MySQL Server: 
File Record inserted. ID: 4


In [4]:
####   OBTAIN   the file using bash commands

import subprocess
import os

# Define your variables
EBSPath = "/data"
CompressedFileName = f"lichess_db_standard_rated_{File_year}-{File_month}.pgn.zst"
CompressedCompletePath = f"https://database.lichess.org/standard/{CompressedFileName}"

# Ensure the directory exists (wget -P usually creates it, but this is safer)
os.makedirs(EBSPath, exist_ok=True)

# Use subprocess to run the bash wget command
# -P specifies the target directory prefix
try:
    # --show-progress: forces the progress bar to appear
    # --progress=bar:force:noscroll: ensures it stays on one line and doesn't spam
    subprocess.run([
        "wget", 
        "-P", EBSPath, 
        "--show-progress", 
        "--progress=bar:force:noscroll", 
        CompressedCompletePath
    ], check=True)
except subprocess.CalledProcessError as e:
    print(f"Download failed: {e}")

DownloadedCompressedFile = "/data/lichess_db_standard_rated_" + File_year + "-" + File_month + ".pgn.zst"
file_size = os.path.getsize(DownloadedCompressedFile)
print(f"Size: {file_size} bytes")



--2026-03-26 03:57:53--  https://database.lichess.org/standard/lichess_db_standard_rated_2013-05.pgn.zst
Resolving database.lichess.org (database.lichess.org)... 141.95.66.62, 2001:41d0:700:5e3e::
Connecting to database.lichess.org (database.lichess.org)|141.95.66.62|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26545457 (25M) [application/octet-stream]
Saving to: ‘/data/lichess_db_standard_rated_2013-05.pgn.zst’

lichess_db_standard  56%[==========>         ]  14.36M  8.65MB/s               

Size: 26545457 bytes


lichess_db_standard 100%[===================>]  25.32M  11.9MB/s    in 2.1s    

2026-03-26 03:57:56 (11.9 MB/s) - ‘/data/lichess_db_standard_rated_2013-05.pgn.zst’ saved [26545457/26545457]



In [5]:
####   DECOMPRESS the file using bash commands
import zstandard as zstd
UnCompressedCompletePath = DownloadedCompressedFile[:-4]

dctx = zstd.ZstdDecompressor()

with open(DownloadedCompressedFile, 'rb') as ifh, open(UnCompressedCompletePath, 'wb') as ofh:
    # copy_stream handles the heavy lifting of reading/writing chunks
    dctx.copy_stream(ifh, ofh)
    
print(f"Decompression finished: {UnCompressedCompletePath}")


Decompression finished: /data/lichess_db_standard_rated_2013-05.pgn


In [6]:
### PARSE the uncompressed file
import parse as pa
metric = []
successbit = False

metric,successbit = pa.Parse(UnCompressedCompletePath,cursor)
print("Total Events Ingested: ",metric[0])
print("Total Events Considered: ",metric[1])
print("Log results: ",metric[2])


LinesTotalRead:  100000
LinesTotalRead:  200000
LinesTotalRead:  300000
LinesTotalRead:  400000
LinesTotalRead:  500000
LinesTotalRead:  600000
LinesTotalRead:  700000
LinesTotalRead:  800000
LinesTotalRead:  900000
LinesTotalRead:  1000000
LinesTotalRead:  1100000
LinesTotalRead:  1200000
LinesTotalRead:  1300000
LinesTotalRead:  1400000
LinesTotalRead:  1500000
LinesTotalRead:  1600000
LinesTotalRead:  1700000
LinesTotalRead:  1800000
LinesTotalRead:  1900000
LinesTotalRead:  2000000
LinesTotalRead:  2100000
LinesTotalRead:  2200000
LinesTotalRead:  2300000
LinesTotalRead:  2400000
LinesTotalRead:  2500000
LinesTotalRead:  2600000
LinesTotalRead:  2700000
LinesTotalRead:  2800000
LinesTotalRead:  2900000
LinesTotalRead:  3000000
LinesTotalRead:  3100000
LinesTotalRead:  3200000
Total Events Ingested:  49594
Total Events Considered:  129956
Log results:  End of file reached.File Parse ended <br>


In [7]:
### Update file details in SQL server

print("Connected to MySQL Server: ")
fs.files_update(Files_id,metric[0],metric[1],file_size,cursor)



Connected to MySQL Server: 
Record updated.


In [8]:
### CLEANUP  the drive

import cleanup as cu
print("Starting Cleanup:")
storageRemain = cu.Cleanup("",DownloadedCompressedFile,UnCompressedCompletePath)
print("Output storageRemain: ",storageRemain)
print("Starting Cleanup:")

Starting Cleanup:
Output storageRemain:  29903.9765625
Starting Cleanup:
